# Spark Catalog
- It is an interface that provides metadata management for Spark SQL. 
- It allows you to interact with and manage databases, tables, views, functions, and cached data within a SparkSession.
- Think of it as a centralized metadata repository for structured data processing.

## Key Components of the Catalog
The Catalog manages the following:
- Databases: Namespaces for organizing tables/views.
- Tables: Structured data sources (e.g., Hive tables, Parquet files).
- Views: Virtual tables defined by SQL queries.
- Functions: User-defined functions (UDFs) or built-in functions.
- Cached Data: Tables/DataFrames cached in memory.

## Accessing the spark catalog


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("catalog_application").getOrCreate()
catalog = spark.catalog


# Common catalog operation

## 1. List Database

In [0]:
# list all database
databases = catalog.listDatabases()
display(databases)

name,catalog,description,locationUri
default,spark_catalog,Default Hive database,dbfs:/user/hive/warehouse


## 2. List table/view in Database


In [0]:
# cleaning the table if present
spark.sql("DROP TABLE IF EXISTS employees")

try:
    if dbutils.fs.ls("/user/hive/warehouse/employees/"):
        print("folder present")
        dbutils.fs.rm("/user/hive/warehouse/employees/", True)
except Exception as e:
    print("folder not present")

folder not present


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS default.employees (
  id INT,
  name STRING,
  salary DOUBLE
)
""")
table = catalog.listTables()
print(table)

[Table(name='employees', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='temp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


## 3. Create / Drop a Database

In [0]:
# create new database
spark.sql("CREATE DATABASE IF NOT EXISTS VISHVA_DB")

# Drop a database (cascade=True drops tables inside)
spark.sql("DROP DATABASE IF EXISTS VISHVA_DB")

Out[44]: DataFrame[]

## 4. Register Tables/Views

In [0]:
df = spark.read.csv("dbfs:/FileStore/tables/employee_data.csv", header=True)
df.createOrReplaceTempView("temp_view")

# Check if the view exists
print("temp_view" in [table.name for table in catalog.listTables()])  # True

True


## 5. Manage Cached Data

In [0]:
catalog.cacheTable("temp_view")

print(catalog.isCached("temp_view"))

catalog.uncacheTable("temp_view")

True


## 6. List Functions

In [0]:
# List all registered functions (built-in + UDFs)
functions = catalog.listFunctions()
for func in functions:
    print(func.name)

!
!=
%
&
*
+
-
/
<
<=
<=>
<>
=
==
>
>=
^
abs
acos
acosh
add_months
aes_decrypt
aes_encrypt
aggregate
and
any
any_value
approx_count_distinct
approx_percentile
approx_top_k
array
array_agg
array_append
array_compact
array_contains
array_distinct
array_except
array_intersect
array_join
array_max
array_min
array_position
array_remove
array_repeat
array_size
array_sort
array_union
arrays_overlap
arrays_zip
ascii
asin
asinh
assert_true
atan
atan2
atanh
avg
base64
between
bigint
bin
binary
bit_and
bit_count
bit_get
bit_length
bit_or
bit_reverse
bit_xor
bool_and
bool_or
boolean
bround
btrim
cardinality
case
cast
cbrt
ceil
ceiling
char
char_length
character_length
charindex
chr
cloud_files
coalesce
collect_list
collect_set
concat
concat_ws
contains
conv
corr
cos
cosh
cot
count
count_if
count_min_sketch
covar_pop
covar_samp
crc32
csc
cume_dist
curdate
current_catalog
current_database
current_date
current_metastore
current_oauth_custom_identity_claim
current_schema
current_timestamp
current_time

# full workflow

In [0]:
# Create a database
spark.sql("CREATE DATABASE IF NOT EXISTS VISHVA_DB")

# Switch to the database
spark.sql("USE VISHVA_DB")

# Read data and register as a table
employee_df = spark.read.csv("dbfs:/FileStore/tables/employee_data.csv", header=True)
employee_df.createOrReplaceTempView("employee_data")

# List tables in the current database
print(catalog.listTables())  # Shows "orders"

# Cache the table
catalog.cacheTable("employee_data")

# Check if cached
print(catalog.isCached("employee_data"))  # True

# Drop the table (temp views are session-scoped)
catalog.dropTempView("employee_data")

# Drop the database
spark.sql("DROP DATABASE IF EXISTS VISHVA_DB")

[Table(name='employee_data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='temp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]
True
Out[50]: DataFrame[]